In [14]:
#pip install pytest-playwright

In [15]:
#!playwright install

In [16]:
#pip install playwright

In [59]:
#pip install python-dotenv

##### 1. Import Utils

In [135]:
import json
import asyncio
from playwright.async_api import async_playwright
from parsel import Selector
from dotenv import load_dotenv
import os

##### 2. Core scrapper

In [136]:
load_dotenv(".env")
RG_EMAIL = os.getenv("RG_USER")
RG_PASSWORD = os.getenv("RG_PASS")

async def rg_login(page):
    print("🔐 Navigating to ResearchGate login...")
    await page.goto("https://www.researchgate.net/login")

    await page.wait_for_timeout(1200)

    # Type like a human
    await page.click('input[name="login"]')
    await page.type('input[name="login"]', RG_EMAIL, delay=90)

    await page.click('input[name="password"]')
    await page.type('input[name="password"]', RG_PASSWORD, delay=100)

    await page.wait_for_timeout(800)

    print("➡ Submitting login form...")
    await page.click('button[type="submit"]')

    # WAIT for profile icon
    try:
        await page.wait_for_selector("a[href*='profile']", timeout=15000)
        print("✅ Login successful!\n")
    except:
        print("❌ Login failed — CAPTCHA or bot-protection encountered")
        html = await page.content()
        print("⚠️ Debug: page contains CAPTCHA?", "captcha" in html.lower())
        raise


In [137]:
CUSTOM_UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"

async def manual_login_capture():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)

        context = await browser.new_context(
            user_agent=CUSTOM_UA,
            viewport={"width": 1400, "height": 900},
            locale="en-US"
        )

        page = await context.new_page()

        print("🔐 Browser opened. Logging in manually is required...")
        await page.goto("https://www.researchgate.net/login")

        print("👉 After successful login, press ENTER here.")
        input()

        await context.storage_state(path="rg_state.json")
        print("✅ Saved authenticated state into rg_state.json")

        await browser.close()

import asyncio
await manual_login_capture()

🔐 Browser opened. Logging in manually is required...
👉 After successful login, press ENTER here.
✅ Saved authenticated state into rg_state.json


In [138]:
BASE = "https://www.researchgate.net"

async def extract_answers(page):
    """Extract all answers from question page"""
    html = await page.content()
    sel = Selector(text=html)
    answers = []
    print(sel)

    answer_items = sel.css(".nova-legacy-v-activity-list-item__body")
    print(answer_items)

    for item in answer_items:
        parts = item.css(".redraft-text::text").getall()
        cleaned = " ".join([p.strip() for p in parts if p.strip()])
        if cleaned:
            answers.append(cleaned)
    return answers


async def scrape_researchgate_questions(query: str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/101.0.4951.64 Safari/537.36"
        )
        # browser = await p.chromium.launch(headless=False)
        # context = await browser.new_context(storage_state="rg_state.json")
        # page = await context.new_page()

        # await rg_login(page)
        
        results = []
        page_num = 1

        while True:
            search_url = f"{BASE}/search/question?q={query}&page={page_num}"
            print(f"\n🔎 Loading search page {page_num}: {search_url}")
            await page.goto(search_url)

            sel = Selector(text=await page.content())
            cards = sel.css(".nova-legacy-c-card__body--spacing-inherit")

            if not cards:
                print("❌ No more results or structure changed.")
                break

            print(f"➡ Found {len(cards)} questions on page {page_num}")

            # Extract each question on this search page
            for card in cards:

                # Skip empty cards
                title = card.css(".nova-legacy-v-entity-item__title a::text").get()
                link = card.css(".nova-legacy-v-entity-item__title a::attr(href)").get()
                if not title or not link:
                    continue

                link = BASE + "/" + link.lstrip("/")
                link = link.split("?")[0]
                print(link)

                snippet_parts = card.css(".redraft-text span::text").getall()
                snippet = " ".join([s.strip() for s in snippet_parts if s.strip()])

                #print(f"\n- Opening question: {title}")
                # await page.goto(link)
                # answers = await extract_answers(page)
                
                question_page = await browser.new_page()
                await question_page.goto(link)
                answers = await extract_answers(question_page)
                await question_page.close()


                results.append({
                    "title": title.strip(),
                    "link": link,
                    "snippet": snippet,
                    "answers": answers
                })
                break

            # Detect pagination: RG has "Next" disabled by removing link
            next_button = sel.css("a[rel='next']")
            if not next_button:
                print("✅ No NEXT page → stopping pagination.")
                break

            page_num += 1
            break
        
        await page.wait_for_timeout(8000)
        # await browser.close()

        print("\nDONE. Total Q&A scraped:", len(results))
        print(json.dumps(results, indent=2, ensure_ascii=False))
        return results

In [139]:
result = await scrape_researchgate_questions("metagenomic")


🔎 Loading search page 1: https://www.researchgate.net/search/question?q=metagenomic&page=1
➡ Found 10 questions on page 1
https://www.researchgate.net/post/Metagenomic_Downstream_Analysis


CancelledError: 

In [106]:
result

[]

In [23]:
async def scrape_researchgate_questions(query: str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, slow_mo=50)
        page = await browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/101.0.4951.64 Safari/537.36")

        questions = []
        page_num = 1

        while True:
            url = f"https://www.researchgate.net/search/question?q={query}&page={page_num}"
            print("Loading:", url)
            await page.goto(url)

            selector = Selector(text=await page.content())

            items = selector.css(".nova-legacy-c-card__body--spacing-inherit")
            if not items:
                print("No results found or structure changed.")
                break

            for question in items:
                print(question)
                title = question.css(".nova-legacy-e-link .nova-legacy-e-link--color-inherit .nova-legacy-e-link--theme-bare").get()
                print("title: ", title)
                if title:
                    title = title.title().strip()

                title_link = question.css(".nova-legacy-v-question-item__title .nova-legacy-e-link--theme-bare::attr(href)").get()
                print(title_link)
                if title_link:
                    title_link = f"https://www.researchgate.net{title_link}"

                snippet = question.css(".redraft-text").xpath("normalize-space()").get()
                question_type = question.css(".nova-legacy-v-question-item__badge::text").get()
                question_date = question.css(".nova-legacy-v-question-item__meta-data-item:nth-child(1) span::text").get()

                views = question.css(".nova-legacy-v-question-item__metrics-item:nth-child(1) .nova-legacy-e-link--theme-bare::text").get()
                views_link = question.css(".nova-legacy-v-question-item__metrics-item:nth-child(1) .nova-legacy-e-link--theme-bare::attr(href)").get()
                if views_link:
                    views_link = f"https://www.researchgate.net{views_link}"

                answer = question.css(".nova-legacy-v-question-item__metrics-item+ .nova-legacy-v-question-item__metrics-item .nova-legacy-e-link--theme-bare::text").get()
                answer_link = question.css(".nova-legacy-v-question-item__metrics-item+ .nova-legacy-v-question-item__metrics-item .nova-legacy-e-link--theme-bare::attr(href)").get()
                if answer_link:
                    answer_link = f"https://www.researchgate.net{answer_link}"

                questions.append({
                    "title": title,
                    "link": title_link,
                    "snippet": snippet,
                    "question_type": question_type,
                    "question_date": question_date,
                    "views": {
                        "views_count": views,
                        "views_link": views_link
                    },
                    "answer": {
                        "answer_count": answer,
                        "answers_link": answer_link
                    }
                })
                break

            print(f"Done page {page_num}")

            # look for next page button (if disabled, break)
            next_disabled = selector.css(".nova-legacy-c-button-group__item:nth-child(9) a::attr(rel)").get()
            if next_disabled:
                break

            page_num += 1
            break

        await browser.close()
        print(json.dumps(questions, indent=2, ensure_ascii=False))

# Run in Jupyter:
await scrape_researchgate_questions("metagenomic")


Loading: https://www.researchgate.net/search/question?q=metagenomic&page=1
<Selector query="descendant-or-self::*[@class and contains(concat(' ', normalize-space(@class), ' '), ' nova-legacy-c-card__body--spacing-inherit ')]" data='<div class="nova-legacy-c-card__body ...'>
title:  None
None
Done page 1
[
  {
    "title": null,
    "link": null,
    "snippet": "What are the best webserver/online tools for Downstream Analysis of 16S amplicon Metagenomic dataset besides Microbiomeanalyst ??",
    "question_type": null,
    "question_date": null,
    "views": {
      "views_count": null,
      "views_link": null
    },
    "answer": {
      "answer_count": null,
      "answers_link": null
    }
  }
]
